# 07 — Forecast Calibration Evaluation

## Purpose
This notebook evaluates probabilistic forecast calibration and compares innovation models using multiple calibration and scoring metrics.

## Outputs
- Calibration summary tables
- Coverage diagnostics
- PIT diagnostics
- Calibration rankings and model comparisons

## Future Work
- Add robustness and stress-testing analysis
- Evaluate calibration under distributional shift

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.append(str(ROOT))

import numpy as np
import pandas as pd

from src.utils.seeds import set_seed

from src.experiments.config import ExperimentConfig
from src.experiments.paths import result_dirs
from src.experiments.artifacts import (
    save_table,
    save_json,
    save_current_figure,
)

from src.api.forecasts import load_forecast_store

set_seed(123)

config = ExperimentConfig()

dgp_names = config.dgp_names

forecast_models = [
    "VAR",
    "RNN",
]

innovation_model_names = [
    "gaussian",
    "bootstrap",
    "student_t",
    "diffusion",
]

dirs = result_dirs(
    "07_calibration_evaluation",
    test=False,
)

forecast_dir = (
    ROOT
    / "results"
    / "forecasts"
    / "06_forecasts"
)

interval = (0.05, 0.95)
nominal_levels = config.nominal_levels

forecast_store = load_forecast_store(
    forecast_dir=forecast_dir,
    forecast_models=forecast_models,
    dgp_names=dgp_names,
    innovation_model_names=innovation_model_names,
)

VAR gaussian gaussian (250, 40, 3) (40, 3)
VAR gaussian bootstrap (250, 40, 3) (40, 3)
VAR gaussian student_t (250, 40, 3) (40, 3)
VAR gaussian diffusion (250, 40, 3) (40, 3)
VAR student_t gaussian (250, 40, 3) (40, 3)
VAR student_t bootstrap (250, 40, 3) (40, 3)
VAR student_t student_t (250, 40, 3) (40, 3)
VAR student_t diffusion (250, 40, 3) (40, 3)
VAR mixture gaussian (250, 40, 3) (40, 3)
VAR mixture bootstrap (250, 40, 3) (40, 3)
VAR mixture student_t (250, 40, 3) (40, 3)
VAR mixture diffusion (250, 40, 3) (40, 3)
VAR heteroskedastic gaussian (250, 40, 3) (40, 3)
VAR heteroskedastic bootstrap (250, 40, 3) (40, 3)
VAR heteroskedastic student_t (250, 40, 3) (40, 3)
VAR heteroskedastic diffusion (250, 40, 3) (40, 3)
RNN gaussian gaussian (250, 40, 3) (40, 3)
RNN gaussian bootstrap (250, 40, 3) (40, 3)
RNN gaussian student_t (250, 40, 3) (40, 3)
RNN gaussian diffusion (250, 40, 3) (40, 3)
RNN student_t gaussian (250, 40, 3) (40, 3)
RNN student_t bootstrap (250, 40, 3) (40, 3)
RNN stud

In [2]:
n_loaded = sum(
    len(forecast_store[forecast_model][dgp_name])
    for forecast_model in forecast_models
    for dgp_name in dgp_names
)

print("Loaded forecast objects:", n_loaded)

Loaded forecast objects: 32


## Main Forecast Evaluation Table

The first evaluation step computes a unified set of probabilistic forecast metrics for each DGP and innovation model.

Metrics include:

- empirical interval coverage,
- average interval width,
- expected calibration error,
- PIT deviation from uniformity,
- CRPS,
- energy score,
- interval score.

Lower is better for ECE, PIT deviation, CRPS, energy score, and interval score. Coverage should be close to the nominal level.

In [3]:
from src.api.calibration import calibration_results_table

main_results_df, summary_objects = calibration_results_table(
    forecast_store=forecast_store,
    forecast_models=forecast_models,
    dgp_names=dgp_names,
    innovation_model_names=innovation_model_names,
    interval=interval,
    nominal_levels=nominal_levels,
)

save_table(
    main_results_df,
    dirs["tables"] / "main_calibration_results.csv",
)

main_results_df

,dgp,forecast_model,innovation_model,avg_coverage,avg_width,energy_score,crps,interval_score,ece,pit_deviation,coverage_1,width_1,coverage_2,width_2,coverage_3,width_3,nominal_coverage,coverage_error,abs_coverage_error
16,gaussian,RNN,gaussian,0.758333,3.052221,1.482231,0.750957,5.873944,0.141667,0.040000,0.725,3.171341,0.700,2.834735,0.850,3.150586,0.9,-1.416667e-01,1.416667e-01
17,gaussian,RNN,bootstrap,0.750000,3.034429,1.492662,0.756141,5.928746,0.166667,0.038333,0.750,3.138819,0.700,2.903046,0.800,3.061423,0.9,-1.500000e-01,1.500000e-01
19,gaussian,RNN,diffusion,0.750000,2.933905,1.522418,0.770316,6.456717,0.177778,0.050000,0.725,2.953744,0.700,2.812114,0.825,3.035857,0.9,-1.500000e-01,1.500000e-01
18,gaussian,RNN,student_t,0.725000,2.939581,1.504756,0.763754,6.249221,0.202778,0.051667,0.675,3.070479,0.675,2.705662,0.825,3.042602,0.9,-1.750000e-01,1.750000e-01
0,gaussian,VAR,gaussian,0.850000,3.694376,1.429168,0.716371,5.700632,0.030556,0.018333,0.875,3.933011,0.800,3.564993,0.875,3.585124,0.9,-5.000000e-02,5.000000e-02
1,gaussian,VAR,bootstrap,0.875000,3.692800,1.437312,0.718980,5.536117,0.041667,0.023333,0.900,3.960802,0.850,3.592302,0.875,3.525295,0.9,-2.500000e-02,2.500000e-02
3,gaussian,VAR,diffusion,0.858333,3.785653,1.463553,0.733258,5.760874,0.044444,0.023333,0.900,3.987017,0.800,3.733162,0.875,3.636780,0.9,-4.166667e-02,4.166667e-02
2,gaussian,VAR,student_t,0.833333,3.588234,1.432465,0.716332,5.670358,0.052778,0.023333,0.875,3.876994,0.775,3.433472,0.850,3.454237,0.9,-6.666667e-02,6.666667e-02
31,heteroskedastic,RNN,diffusion,0.833333,6.243694,2.276519,1.108048,11.284266,0.047222,0.020000,0.825,6.036653,0.850,6.821125,0.825,5.873304,0.9,-6.666667e-02,6.666667e-02
29,heteroskedastic,RNN,bootstrap,0.875000,6.656965,2.254866,1.104373,10.438031,0.052778,0.023333,0.875,6.845672,0.875,6.228867,0.875,6.896355,0.9,-2.500000e-02,2.500000e-02


In [4]:
display_cols = [
    "forecast_model",
    "dgp",
    "innovation_model",
    "avg_coverage",
    "abs_coverage_error",
    "avg_width",
    "ece",
    "pit_deviation",
    "crps",
    "energy_score",
    "interval_score",
]

compact_results_df = main_results_df[
    display_cols
].copy()

compact_results_df

,forecast_model,dgp,innovation_model,avg_coverage,abs_coverage_error,avg_width,ece,pit_deviation,crps,energy_score,interval_score
16,RNN,gaussian,gaussian,0.758333,1.416667e-01,3.052221,0.141667,0.040000,0.750957,1.482231,5.873944
17,RNN,gaussian,bootstrap,0.750000,1.500000e-01,3.034429,0.166667,0.038333,0.756141,1.492662,5.928746
19,RNN,gaussian,diffusion,0.750000,1.500000e-01,2.933905,0.177778,0.050000,0.770316,1.522418,6.456717
18,RNN,gaussian,student_t,0.725000,1.750000e-01,2.939581,0.202778,0.051667,0.763754,1.504756,6.249221
0,VAR,gaussian,gaussian,0.850000,5.000000e-02,3.694376,0.030556,0.018333,0.716371,1.429168,5.700632
1,VAR,gaussian,bootstrap,0.875000,2.500000e-02,3.692800,0.041667,0.023333,0.718980,1.437312,5.536117
3,VAR,gaussian,diffusion,0.858333,4.166667e-02,3.785653,0.044444,0.023333,0.733258,1.463553,5.760874
2,VAR,gaussian,student_t,0.833333,6.666667e-02,3.588234,0.052778,0.023333,0.716332,1.432465,5.670358
31,RNN,heteroskedastic,diffusion,0.833333,6.666667e-02,6.243694,0.047222,0.020000,1.108048,2.276519,11.284266
29,RNN,heteroskedastic,bootstrap,0.875000,2.500000e-02,6.656965,0.052778,0.023333,1.104373,2.254866,10.438031


In [5]:
from src.api.calibration import relative_improvement_table

relative_improvement_df = relative_improvement_table(
    main_results_df=main_results_df,
    forecast_models=forecast_models,
    dgp_names=dgp_names,
    innovation_model_names=innovation_model_names,
    baseline_model="gaussian",
)

save_table(
    relative_improvement_df,
    dirs["tables"] / "relative_improvement_vs_gaussian.csv",
)

relative_improvement_df

,forecast_model,dgp,innovation_model,ece_relative_improvement,pit_deviation_relative_improvement,crps_relative_improvement,energy_score_relative_improvement,interval_score_relative_improvement,abs_coverage_error_relative_improvement
0,VAR,gaussian,gaussian,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000e+00
1,VAR,gaussian,bootstrap,-0.363636,-2.727273e-01,-0.003642,-0.005698,0.028859,5.000000e-01
2,VAR,gaussian,student_t,-0.727273,-2.727273e-01,0.000054,-0.002307,0.005311,-3.333333e-01
3,VAR,gaussian,diffusion,-0.454545,-2.727273e-01,-0.023573,-0.024059,-0.010568,1.666667e-01
4,VAR,student_t,gaussian,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000e+00
5,VAR,student_t,bootstrap,0.700000,2.608696e-01,0.037792,0.034872,-0.003418,-7.505999e+13
6,VAR,student_t,student_t,0.600000,0.000000e+00,0.004172,0.003535,0.011110,0.000000e+00
7,VAR,student_t,diffusion,0.600000,4.347826e-02,0.009248,0.006275,0.006893,0.000000e+00
8,VAR,mixture,gaussian,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000e+00
9,VAR,mixture,bootstrap,-1.352941,-5.000000e-01,-0.013586,-0.012481,-0.051302,-2.857143e-01


In [6]:
from src.api.calibration import headline_ranking_table

headline_ranking_df = headline_ranking_table(main_results_df=main_results_df,forecast_models=forecast_models,dgp_names=dgp_names,)
save_table(headline_ranking_df,dirs["tables"] / "headline_rankings.csv",)
headline_ranking_df

,forecast_model,dgp,best_ece_innovation,best_ece_value,best_pit_deviation_innovation,best_pit_deviation_value,best_crps_innovation,best_crps_value,best_energy_score_innovation,best_energy_score_value,best_interval_score_innovation,best_interval_score_value,best_abs_coverage_error_innovation,best_abs_coverage_error_value
0,VAR,gaussian,gaussian,0.030556,gaussian,0.018333,student_t,0.716332,gaussian,1.429168,bootstrap,5.536117,bootstrap,2.500000e-02
1,VAR,student_t,bootstrap,0.008333,bootstrap,0.028333,bootstrap,0.836386,bootstrap,1.679683,student_t,7.220528,diffusion,1.110223e-16
2,VAR,mixture,gaussian,0.047222,gaussian,0.033333,gaussian,1.152898,gaussian,2.299282,gaussian,10.535329,gaussian,5.833333e-02
3,VAR,heteroskedastic,bootstrap,0.041667,bootstrap,0.036667,student_t,1.129142,student_t,2.301139,diffusion,10.935295,bootstrap,3.333333e-02
4,RNN,gaussian,gaussian,0.141667,bootstrap,0.038333,gaussian,0.750957,gaussian,1.482231,gaussian,5.873944,gaussian,1.416667e-01
5,RNN,student_t,gaussian,0.027778,diffusion,0.026667,diffusion,0.846077,diffusion,1.713617,gaussian,7.561624,gaussian,3.333333e-02
6,RNN,mixture,gaussian,0.119444,gaussian,0.036667,gaussian,1.203337,gaussian,2.399033,gaussian,11.225308,gaussian,1.166667e-01
7,RNN,heteroskedastic,diffusion,0.047222,diffusion,0.020000,bootstrap,1.104373,bootstrap,2.254866,bootstrap,10.438031,bootstrap,2.500000e-02


In [7]:
headline_ranking_df[["forecast_model",
                     "dgp",
                     "best_ece_innovation","best_ece_value",
                     "best_crps_innovation","best_crps_value",
                     "best_energy_score_innovation","best_energy_score_value",
                     "best_interval_score_innovation","best_interval_score_value",]]

,forecast_model,dgp,best_ece_innovation,best_ece_value,best_crps_innovation,best_crps_value,best_energy_score_innovation,best_energy_score_value,best_interval_score_innovation,best_interval_score_value
0,VAR,gaussian,gaussian,0.030556,student_t,0.716332,gaussian,1.429168,bootstrap,5.536117
1,VAR,student_t,bootstrap,0.008333,bootstrap,0.836386,bootstrap,1.679683,student_t,7.220528
2,VAR,mixture,gaussian,0.047222,gaussian,1.152898,gaussian,2.299282,gaussian,10.535329
3,VAR,heteroskedastic,bootstrap,0.041667,student_t,1.129142,student_t,2.301139,diffusion,10.935295
4,RNN,gaussian,gaussian,0.141667,gaussian,0.750957,gaussian,1.482231,gaussian,5.873944
5,RNN,student_t,gaussian,0.027778,diffusion,0.846077,diffusion,1.713617,gaussian,7.561624
6,RNN,mixture,gaussian,0.119444,gaussian,1.203337,gaussian,2.399033,gaussian,11.225308
7,RNN,heteroskedastic,diffusion,0.047222,bootstrap,1.104373,bootstrap,2.254866,bootstrap,10.438031


In [8]:
wins = []

metrics = [ "ece", "crps", "energy_score","interval_score",]

for metric in metrics:
    counts = ( headline_ranking_df[f"best_{metric}_innovation"].value_counts().to_dict())

    for innovation_model, n in counts.items():
        wins.append( { "metric": metric, "innovation_model": innovation_model,"wins": n,})

win_df = pd.DataFrame(wins)
win_df.sort_values(["metric", "wins"],ascending=[True, False],)

,metric,innovation_model,wins
3,crps,gaussian,3
4,crps,student_t,2
5,crps,bootstrap,2
6,crps,diffusion,1
0,ece,gaussian,5
1,ece,bootstrap,2
2,ece,diffusion,1
7,energy_score,gaussian,4
8,energy_score,bootstrap,2
9,energy_score,student_t,1


In [9]:
win_pivot = win_df.pivot( index="innovation_model", columns="metric", values="wins",).fillna(0)
win_pivot

metric,crps,ece,energy_score,interval_score
innovation_model,,,,
bootstrap,2.0,2.0,2.0,2.0
diffusion,1.0,1.0,1.0,1.0
gaussian,3.0,5.0,4.0,4.0
student_t,2.0,0.0,1.0,1.0


Diffusion innovation models provide the greatest benefit when
innovation distributions exhibit complex non-Gaussian structure
such as mixtures and heteroskedasticity.

The largest gains appear in forecast distribution quality
(energy score and interval score), while improvements in
calibration metrics such as ECE are more context dependent.

Bootstrap:
better calibration

Diffusion:
better probabilistic forecasts